## Data Update for 2025-07-28 export - Cleaning the Data

This file is to clean the New England Whalers data downloaded from the Whaling logbook database.

The main layout of this script is as follows:
1. Import the necessary packages and data, and drop any unnecessary columns and rows
2. Clean the dataset, as follows:
    - REMOVE DUPLICATE ENTRIES
    - STANDARDIZE COLUMNS WITH EXTRA TEXT AND MAPPING WIND TERMS TO BF SCALE
    - CATCH/CORRECT COORDINATE ERRORS
    - PLOTTING TRAJECTORIES TO CATCH MISSED ERRORS
    - EXPORT CLEANED DATASET

In [ ]:
# Import
import numpy as np
import pandas as pd
import xarray as xr # not used in this script
import matplotlib.pyplot as plt
import datetime, os
from cartopy import crs as ccrs
from cartopy import feature as cfeature
import seaborn as sns # not used in this script
from tabulate import tabulate # not used in this script
import sys
from matplotlib.colors import Normalize # not used in this script
import re
from IPython.display import clear_output
import datetime
import textwrap
import csv

pd.options.display.max_columns = 50

print("Last updated on {}".format(datetime.datetime.now().ctime()))

In [9]:
# Setting up a folder system to save files and figures to 
# Create folders called "Figures", "Text Files", and "CSV Files" manually to save everything to 

# Get current directory
current_directory = os.getcwd()
#print(current_directory)

# Specify the path to the folder you want to save data to 
Figures = os.path.join(current_directory, 'Figures')
Files = os.path.join(current_directory, 'Text_files')
CSV = os.path.join(current_directory, 'CSV_files')

# Create directories only if they don't exist
os.makedirs(Figures, exist_ok=True)
os.makedirs(Files, exist_ok=True)
os.makedirs(CSV, exist_ok=True)

### Importing Logbook Dataset

In [ ]:
# labels corresponding to missing data
na_values = ['No observation', 'No observations', 'No Observation', 'No Observations',
             'no observation', 'no observations', 'None given', 'none given', 'None Given', 'none',
             'none recorded', 'not recorded', 'None recorded', 'Not given', 'not given', ' ', 'N.A.', 'Na', 'Does not say', 'N.A', 'Deos not say', 'N A']

df = pd.read_csv(os.path.join(CSV, 'logentries-export-2025-07-28.csv'), na_values=na_values, low_memory=False)

# Combine using column names
df['DateTime'] = pd.to_datetime(df['Entry Date'] + ' ' + df['Local Time'], errors='coerce')
df.columns

In [11]:
# # Adding researchers to the df to look into how new entries are going
# researchers_df = pd.read_csv('researchers-export-2025-03-31.csv',
#                              na_values=na_values)

# # Merge on the 'ID' column, adding the 'researcher' column
# df = df.merge(researchers_df[['ID', 'Researcher', 'Published']], on='ID', how='left')

In [ ]:
#How many entries since last export?
#Need to update subtracted number every entry
print(f"There have been {len(df) - 117849} entries since the lasts export") #number given by last export summary

In [13]:
# Finding how many times there is a wind force value in the second or third
# observation but not first to see if we want to try and infill with these 

# Create a dataframe when original wind speed/force observation is nan, but 2. and 3. are not nan
df['wind force'] = (df["Wind Speed/Force"].isna() & (df["2. Wind Speed/Force"].notna() | df["3. Wind Speed/Force"].notna()))
df_wf = df[df['wind force']==True]
#df_wf

In [14]:
# Dropping columns not needed
columns_to_drop = ['Current', '2. Ship Heading/Course', '2. Wind Direction', '2. Wind Speed/Force', 
                   '2. Sea State', '2. Cloud Cover', '2. Weather', '3. Ship Heading', '3. Wind Direction', 
                   '3. Wind Speed/Force', '3. Sea State', '3. Cloud Cover', '3. Weather',
                   'Instrumental Observations']

# Drop the specified columns
df.drop(columns=columns_to_drop, inplace=True)

In [16]:
# Check dataframe to make sure unnecessary columns are gone
#df.head

In [ ]:
# Check dataframe to make sure unnecessary columns are gone
df.info()

## Cleaning the Dataset

Dropping 'TEST LOG BOOK NAME' Entries

In [ ]:
df.drop(df.loc[df["LogBook ID"] == "TEST LOG BOOK NAME"].index,axis=0,inplace=True)

Dropping 'Westward-1978' Entries

In [ ]:
#df.drop(df.loc[df["Entry Date Time"].dt.year==2021.].index,axis=0,inplace=True)
df.drop(df.loc[df["LogBook ID"]=="Westward-1978"].index,inplace=True)
df.drop(df.loc[df["LogBook ID"]=="55"].index,inplace=True)

np.shape(df)

Manual Cleanups

In [ ]:
# replace DateTime-strings that end with ' nan' with np.nan
# df.DateTime.str.endswith(' nan') #returns a boolean list of all rows, setting rows that fulfill the .endswith query to True and all others to False.
# df.loc[boolean-list,'DateTime'] accesses the 'DateTime' column of those rows that were set to True. = np.nan sets those values to NaN values.
df.loc[df.DateTime.astype(str).str.endswith(' nan'), 'DateTime'] = np.nan

In [ ]:
# converting 'DateTime' column to actual DateTime and calling it "Entry Date Time"
df['Entry Date Time'] = pd.to_datetime(df.DateTime, format = '%Y-%m-%d %H:%M:%S')
# deleting row "DateTime"
df.drop('DateTime',axis=1)

## Removing repeated data

We have a few ships with multiple logbooks as well as duplicate entries that have verified as intentional (crossing the meridian, noting specific weather, etc.). Here we keep a running list of these so that we only flag new duplicate entries with each new export.  

In [ ]:
# Removing potential duplicates of data from the Leonidas ships

# Filter rows corresponding to "Leonidas (ship) Journal 1830-1833" and "Leonidas (ship) 1830-1833"
leonidas_journal_rows = df[df['LogBook ID'] == 'Leonidas (ship) Journal 1830-1…']
leonidas_rows = df[df['LogBook ID'] == 'Leonidas (ship) 1830-1833']

# Get the 'Entry Date Time' values from "Leonidas (ship) Journal 1830-1833"
journal_entry_times = leonidas_journal_rows['Entry Date Time']

# Find indices of rows from "Leonidas (ship) Journal 1830-1833" that have the same 'Entry Date Time' values as in "Leonidas (ship) 1830-1833"
indices_to_drop = leonidas_journal_rows[leonidas_journal_rows['Entry Date Time'].isin(leonidas_rows['Entry Date Time'])].index

# Drop the "Leonidas (ship) Journal 1830-1833" rows from the original DataFrame
df = df.drop(indices_to_drop)
#df

# This should end up getting rid of all Leonidas Journal entries, we do not need them as 
# they duplicate the other Leonidas official data 

In [ ]:
# converting 'DateTime' column to actual DateTime and calling it "Entry Date Time"
df['Entry Date Time'] = pd.to_datetime(df.DateTime, format = '%Y-%m-%d %H:%M:%S')
# deleting row "DateTime"
df.drop('DateTime', axis=1, inplace=True)

In [ ]:
# Removing potential duplicates of data from the Margaret ships

# Filter rows corresponding to "Margaret (ship) 1835-1836"
margaret_1835_1836_rows = df[df['LogBook ID'] == 'Margaret (ship) 1835-1836']

# Drop rows corresponding to "Margaret (ship) 1835-1836"
df = df.drop(margaret_1835_1836_rows.index)

# Filter rows corresponding to "Margaret (ship) 1835-1838" and "Margaret (Ship) 1835–1838"
margaret_ship_1835_1838_rows = df[df['LogBook ID'] == 'Margaret (ship) 1835-1838']
margaret_Ship_1835_1838_rows = df[df['LogBook ID'] == 'Margaret (Ship) 1835–1838']

# Find indices of rows from "Margaret (ship) 1835-1838" that have the same 'Entry Date Time' values as in "Margaret (Ship) 1835–1838"
indices_to_drop = margaret_ship_1835_1838_rows[margaret_ship_1835_1838_rows['Entry Date Time'].isin(margaret_Ship_1835_1838_rows['Entry Date Time'])].index

# Drop the rows from "Margaret (ship) 1835-1838" that have the same 'Entry Date Time' value as "Margaret (Ship) 1835–1838"
df = df.drop(indices_to_drop)

# Rename remaining entries from 'Margaret (ship) 1835-1838' to match 'Margaret (Ship) 1835-1838'
df['LogBook ID'] = df['LogBook ID'].replace('Margaret (ship) 1835-1838', 'Margaret (Ship) 1835-1838')
#df

# This should get rid of all entries from Margaret 1835-1836 and then keep rows from
# Margaret (ship) where there are dates missing for Margaret (Ship) and delete the rest

### Double Dates

In [ ]:
pres_rows = df[df['LogBook ID'] == 'President (Bark) 1865-1869']
lancer_rows = df[df['LogBook ID'] == 'Lancer (Ship) 1865-1868']

# Combine indices
indices_to_drop = pres_rows.index.union(lancer_rows.index)

# Drop the rows
df = df.drop(index=indices_to_drop)

In [ ]:
#Reading in ok duplicate entries file:
ok_df = pd.read_csv("ok_duplicate_dates.txt", sep="\t")
ok_df['Date'] = pd.to_datetime(ok_df['Date']).dt.date  # ensure 'Date' column is datetime.date objects
ok_set = set(zip(ok_df['LogBook ID'], ok_df['Date']))  # create lookup set

In [ ]:
# Group by logbook and date
date_groups = df.groupby(['LogBook ID', df['Entry Date Time'].dt.date])
duplicate_date_groups = date_groups.filter(lambda group: len(group) > 1)

# Collect duplicate ID pairs
duplicate_pairs = []

# Group and check for new duplicates
for (logbook_id, date), group in duplicate_date_groups.groupby(['LogBook ID', duplicate_date_groups['Entry Date Time'].dt.date]):
    
    # SKIP if this duplicate is already approved
    if (logbook_id, date) in ok_set:
        continue

    ids = group['ID'].unique()
    if len(ids) == 2:  # Only include clean duplicates (exactly 2 entries)
        duplicate_pairs.append(list(ids))
        print(f"Date: {date}, LogBook ID: {logbook_id}, IDs: {ids}")
    else:
        print(f" Skipping group with {len(ids)} entries on {date} ({logbook_id})")

print(f"Total duplicate pairs found: {len(duplicate_pairs)}")

#### New duplicates examined and fixed:
- **Amazon 12/15/1857**: entered twice (identical), removed one of the entries from the db
- **Amazon 1859-10-25**: entered 1x by sahitha and 1x by abigail field, removed abigail entry
- **Amazon 1860-03-30**: entered twice (identical), removed one of the entries from the db
- **Congress (Ship) 1857-1859 1857-01-02**: entered 2x (identical), removed one from db
- **Falcon (Bark) 1865-1867 1866-08-13**: entered 2x (identical), removed one from db
- **Francis Allyn (Schooner) 1891–1893 1893-04-25**: second 4/25 changed to 4/26 (previously skipped to 4/27)
- **Herald (Ship) 1834-1837 1834-07-25**: already corrected in db (must have happened after export)
- **Jason** - reached out to ___ they are fixing
- **Mercator(Ship) 1840-1843 1842-12-10**: second entry due to ship sighting - added to ok dupes
- **Mercator (Ship) 1840-1843 1842-07-16**: entered 2x (identical), removed one from db
- **Mercator (Ship) 1840-1843 1842-05-18**: entered 2x (identical), removed one from db
- **Mercator (Ship) 1840-1843 1841-11-15**: entered 2x (identical), removed one from db
- **President** - need other errors to be corrected
- **Sea Breeze** - reached out to ___ they are fixing
- **Young Phenix (1867 -1871) 1870-03-02**: entered 2x (identical), removed one from db

In [ ]:
# Define log file location
log_file = "duplicate_corrections_log.txt"
log_path = os.path.join(Files, log_file)

# Reset log at start
with open(log_path, "w") as f:
    f.write("=== Log Start ===\n")

In [ ]:
from utils.cleaning import correct_dupes

for i, pair in enumerate(duplicate_pairs, 1):
    clear_output(wait=True)
    print(f"Reviewing {i} of {len(duplicate_pairs)}: IDs {pair}")
    df = correct_dups(df, pair, log_path=log_path)

In [ ]:
# saving corrected df so we dont have to do that again.
outpath = os.path.join(CSV, "logbook_no_dupes.csv")
df.to_csv(outpath, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

### Cleaning page, depth, and ship sightings columns

In [ ]:
current_directory = os.getcwd()
CSV = os.path.join(current_directory, 'CSV Files')
df_path = os.path.join(CSV, "logbook_no_dupes.csv")

df1 = pd.read_csv(df_path)

In [ ]:
#Revised version that avoids deprecation:
# Strip strings and store
page_stripped = df1['Page'].astype(str).str.strip()

# Remove brackets based on condition
df1.loc[page_stripped.str.startswith('('), 'Page'] = page_stripped.str[1:4]
df1.loc[page_stripped.str.endswith(')'), 'Page'] = page_stripped.str[0:1]

In [ ]:
#Revised future-proofed version
with pd.option_context('display.max_rows', None):
    page_stripped = df1['Page'].astype(str).str.strip()

In [ ]:
# Convert common typos to NaN
df1.loc[df1.Page.isin(['N', 'N/a', 'n/a`']), 'Page'] = np.nan

# map string to ints
page_map = {
    '1-8': 1, '3 1/2': 3, '5 1/2': 5, '14-15': 14, '22-23': 22,
    '30-31': 30, '48-49': 48, '94/95': 94, '97-98': 97, '108-109': 108,
    '121 (says 107)': 121, '122-123': 122, '158-159': 158,
    '159-160': 160, '177-178': 177, '186-187': 186,
    '(8)': 8, '(6)': 6, '(4)': 4, '(2)': 2, '(16': 16,
    '(17': 17, '(18': 18, '(19': 19, '(20': 20
}

#execute conversion
df1['Page'] = df1['Page'].replace(page_map)

#cleaning other string add-ons
df1["Page"] = df1["Page"].str.strip('"ab')

# converting 'Page' values to numeric
df1['Page']=pd.to_numeric(df1.Page, errors = 'coerce')

In [ ]:
from utils.cleaning import clean_depth_column

temp_df = df1.copy()
df_dep_cleaned = clean_depth_column(temp_df, column='Depth')

Handling common strings that should be set to NaN for other columns:

In [ ]:
#New df to continue cleaning
df2 = df_dep_cleaned.copy()
keywords = ['none', 'nnone', 'not recorded', 'Not Given']

for vname in ['Ship Heading/Course', 'Wind Direction', 'Wind Speed/Force']:
    # Convert to string to handle NaNs safely
    col = df2[vname].astype(str).str.strip()

    for kw in keywords:
        mask = col.str.startswith(kw)
        df2.loc[mask, vname] = np.nan

In [ ]:
# removing double spaces from 'Ship Sightings', 'Misc', 'Landmarks', and 'Wind Direction' to make sure rows are sorted properly
# Create a unique separator that does not appear in data
separator = '###'

# Replace consecutive spaces with the separator 
df2['Ship Sightings'] = df2['Ship Sightings'].str.replace(r'\s+', separator, regex=True)
df2['Miscellaneous Observations'] = df2['Miscellaneous Observations'].str.replace(r'\s+', separator, regex=True)
df2['Landmark'] = df2['Landmark'].str.replace(r'\s+', separator, regex=True)

# Replace the separator with a single space
df2['Ship Sightings'] = df2['Ship Sightings'].str.replace(separator, ' ')
df2['Miscellaneous Observations'] = df2['Miscellaneous Observations'].str.replace(separator, ' ')
df2['Landmark'] = df2['Landmark'].str.replace(separator, ' ')

### Clean/Standardize Wind Direction

In [ ]:
# Get unique values
unique_directions = df2['Wind Direction'].dropna().unique()

# Save to a text file (one entry per line)
with open('unique_wind_directions.txt', 'w') as f:
    for direction in unique_directions:
        f.write(f"{direction}\n")

Identifying Wind Speed terms in Column "Wind Direction" and updating "Wind Speed/Force" & "Wind Direction" accordingly

In [ ]:
df2.loc[df2['Wind Direction']== 'Calm',"Wind Speed/Force"]    # No need to update wind speed entry
df2.loc[df2['Wind Direction']== 'Light',"Wind Speed/Force"] = 'Light'
df2.loc[df2['Wind Direction']== 'Light airs',"Wind Speed/Force"] = 'Light airs'
df2.loc[df2['Wind Direction']== 'light wind',"Wind Speed/Force"] = 'light wind'
df2.loc[df2['Wind Direction']== 'light wind',"Direction"] = 'from NE'

In [ ]:
from utils.cleaning import clean_wind_dirs

In [ ]:
# #UNCOMMENT CELL TO EXAMINE RESULTS OF clean_wind_dirs

# # copy the full DataFrame so original is untouched
# temp_df = df2.copy()

# # 1) pick 40 random, UNIQUE raw values
# sample_raw = (
#     temp_df['Wind Direction']      # original column
#       .dropna()                    # ignore pre-existing NaNs
#       .unique()                    # keep unique only
# )
# sample_raw = (
#     pd.Series(sample_raw)
#       .sample(n=min(50, len(sample_raw)),
#               random_state=42)     # reproducible sample
# )

# # build a test DataFrame
# sample_df = pd.DataFrame({'Wind Direction': sample_raw.values})

# # clean
# cleaned_df, _ = clean_wind_dirs(sample_df)

# # convert to numeric
# sample_df['Cleaned'] = cleaned_df['Wind Direction']
# sample_df['Bearing'] = wind_dir_to_numeric(cleaned_df)

# # show before / after / numeric
# print(sample_df)

In [ ]:
#apply to all wind cleaning to entire dataframe

temp_df = df2.copy()
wind_cleaned_df, to_nan_vals = clean_wind_dirs(temp_df)

#inspect entries set to NaN and add any that should not be to above mapping dicts
print(len(to_nan_vals))
to_nan_vals

In [ ]:
from utils.cleaning import wind_dir_to_numeric

temp_df = wind_cleaned_df.copy()
numeric_wind_df = wind_dir_to_numeric(temp_df, out_col='WD_Bearing')

#numeric_wind_df

Clean/Standardize Wind Speed/Force

In [ ]:
#applying initial string cleaning
from utils.cleaning import init_wind_force_clean

basic_clean_wf_df = init_wind_force_clean(numeric_wind_df)

Converting wind force to bf scale:

In [ ]:
from utils.cleaning import load_beaufort_map, parse_beaufort_series, basic_clean_wf_df

### Load BF mapping from your txt file
bf_map = load_beaufort_map(wf_txt_file, unique_only=True)

wf_txt_file = os.path.join(Files, "wind_force_classified.txt")
log = os.path.join(Files, "beaufort_additions_log.txt")

# Apply to your DataFrame
outdf = parse_beaufort_series(
    basic_clean_wf_df, 
    col="Wind Speed/Force",
    bf_map=bf_map,
    new_col='BF Value',
    mapping_txt_file=wf_txt_file,   # this will overwrite with updated sections!
    interactive=True,
    log_file=log
)

Clean/standardize sea state, cloud cover, and weather

In [ ]:
from utils.cleaning import clean_remaining_strings

temp_df = outdf.copy()
all_strings_cleaned_df = clean_remaining_strings(temp_df)

outpath = os.path.join(CSV, "logbooks_clean_strings.csv")
all_strings_cleaned_df.to_csv(outpath, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

# Flagging/correcting latitude and longitude columns

Here we are ID'ing instances of: 
- lat/lon values that have no direction
- lat/lon with values larger that 90/180
- lat/lon entries switched
- Lat w E/W (rather than N/S)
- Lon w S/N (rather than E/W)
- lat/lon with ‘Miles’ or ‘miles’ or ‘M’ in it
- other symbols/formats (e.g. DD.MM)

In [ ]:
# import utilities and define paths
from utils.cleaning import (
    normalize_coords,
    flag_and_convert_miles,
    flag_coords_too_many_digits,
    correct_coord,
    flag_coords_missing_direction,
    flag_direction_symbol_errors,
    batch_correct_coords,
    apply_coord_corrections,
    flag_coords_beyond_bounds,
    examine_and_correct_outliers,
    save_invalid_coords,
)

import os
import pandas as pd

current_directory = os.getcwd()
Figures = os.path.join(current_directory, 'Figures')
Files   = os.path.join(current_directory, 'Text Files')
CSV     = os.path.join(current_directory, 'CSV Files')
os.makedirs(Figures, exist_ok=True)
os.makedirs(Files, exist_ok=True)
os.makedirs(CSV, exist_ok=True)

In [ ]:
# load / prepare
df_path = os.path.join(CSV, "logbooks_clean_strings.csv")
df = pd.read_csv(df_path, low_memory=False)

# ensure Entry Date is datetime
if not pd.api.types.is_datetime64_any_dtype(df['Entry Date']):
    df['Entry Date'] = pd.to_datetime(df['Entry Date'], errors='coerce')

# sort for nice context windows
df = df.sort_values(['LogBook ID', 'Entry Date']).reset_index(drop=True)

In [ ]:
# normalize formatting and convert text to DMS
temp_df = df.copy()
file = os.path.join(Files, "coords_normalized.txt")

df_1 = normalize_coords(temp_df, lat_col='Latitude', lon_col='Longitude', verbose=False, save_fixes_file=file)

In [ ]:
# convert special text like Miles/Equator/etc
df_2 = flag_and_convert_miles(df_1)

In [ ]:
# correct entries with too-many-digits
df_3 = batch_correct_coords(df_2, flag_coords_too_many_digits)

In [ ]:
# correct entries missing direction or wrong symbol
outpath = os.path.join(Files, "coord_corrections_log.txt")

# df_4 = batch_correct_coords(df_3, flag_coords_missing_direction, log_path=outpath)
# df_5 = batch_correct_coords(df_4, flag_direction_symbol_errors, log_path=outpath)

#HARDCODING CHANGES (MADE IN INPUTS AND SAVED TO COORDS_NORMALIZED.TXT) TO SAVE TIME
import numpy as np

# Dictionary mapping ID → {column: new_value}
hardcoded_changes = {
    122079: {"Longitude": "110 00 E"},
    122921: {"Longitude": "119 52 E"},
    122935: {"Longitude": np.nan},
    123201: {"Longitude": "110 00 E"},
    121775: {"Latitude": "28 40 N", "Longitude": "110 15 E"},
    123818: {"Latitude": "15 02 N"},
    124079: {"Longitude": "173 43 W"},
    124080: {"Longitude": "175 30 W"},
    125014: {"Longitude": "159 55 E"},
    125016: {"Longitude": "159 55 E"},
    125017: {"Longitude": "159 46 E"},
    125018: {"Longitude": "159 36 E"},
    124758: {"Latitude": "00 50 N"},
    125755: {"Longitude": "175 40 E"},
    122940: {"Longitude": "127 00 E"},
    124065: {"Latitude": "37 02 N"},
    124436: {"Latitude": "30 28 N", "Longitude": "173 26 W"},
    122872: {"Latitude": "35 48 N", "Longitude": "126 58 E"},
}

# Apply changes
df_5 = df_3.copy()
for row_id, changes in hardcoded_changes.items():
    for col, val in changes.items():
        df_5.loc[df["ID"] == row_id, col] = val

In [ ]:
# apply known manual corrections (lon > 180 recorded style, etc.)
COORD_FIXES = {
    # Good Return (1841–1844)
    48751: {"Longitude": "128 38 W"}, 48749: {"Longitude": "128 10 W"}, 48747: {"Longitude": "128 55 W"},
    48744: {"Longitude": "128 45 W"}, 48743: {"Longitude": "128 45 W"}, 48740: {"Longitude": "129 00 W"},
    48739: {"Longitude": "129 30 W"}, 48738: {"Longitude": "130 45 W"}, 48737: {"Longitude": "133 00 W"},
    48736: {"Longitude": "133 15 W"}, 48735: {"Longitude": "133 35 W"}, 48734: {"Longitude": "133 30 W"},
    48733: {"Longitude": "135 00 W"}, 48732: {"Longitude": "157 00 W"}, 48731: {"Longitude": "159 20 W"},
    48724: {"Longitude": "161 30 W"}, 48721: {"Longitude": "161 30 W"}, 48714: {"Longitude": "162 52 W"},
    48712: {"Longitude": "164 46 W"}, 48709: {"Longitude": "167 30 W"}, 48708: {"Longitude": "170 30 W"},
    48703: {"Longitude": "173 00 W"}, 48699: {"Longitude": "176 30 W"}, 48697: {"Longitude": "179 00 W"},

    # Gideon Howland (Ship) 1838–1842
    102329: {"Longitude": "170 13 W"}, 102163: {"Longitude": "155 20 W"}, 102161: {"Longitude": "159 00 W"},
    102160: {"Longitude": "161 00 W"}, 102159: {"Longitude": "161 13 W"}, 102158: {"Longitude": "163 33 W"},
    102157: {"Longitude": "165 13 W"}, 102156: {"Longitude": "166 32 W"}, 102155: {"Longitude": "168 04 W"},
    102154: {"Longitude": "169 02 W"}, 102153: {"Longitude": "168 09 W"}, 102152: {"Longitude": "168 14 W"},
    102151: {"Longitude": "169 12 W"}, 102150: {"Longitude": "168 46 W"}, 102149: {"Longitude": "168 48 W"},
    102148: {"Longitude": "168 42 W"}, 102144: {"Longitude": "171 12 W"}, 102143: {"Longitude": "171 18 W"},
    102142: {"Longitude": "171 28 W"}, 102134: {"Longitude": "170 26 W"}, 102131: {"Longitude": "170 59 W"},
    102125: {"Longitude": "177 17 W"}, 102114: {"Longitude": "171 58 W"}, 102111: {"Longitude": "171 36 W"},
    102108: {"Longitude": "172 49 W"}, 102102: {"Longitude": "173 19 W"}, 102100: {"Longitude": "174 15 W"},
    101999: {"Longitude": "174 15 W"}, 101997: {"Longitude": "175 51 W"}, 101953: {"Longitude": "177 13 W"},
    101950: {"Longitude": "178 51 W"}, 101947: {"Longitude": "179 51 W"},

    # President (ship)
    113588: {"Longitude": "176 82 W"},
}

df_6 = apply_coord_corrections(df_5, COORD_FIXES)

In [ ]:
# fFlag out-of-bounds entrues and correct
flagged_ids_lon, flagged_ids_lat = flag_coords_beyond_bounds(df_6)

df_7 = examine_and_correct_outliers(df_6, flagged_ids_lon, col='Longitude', log_path=outpath)
df_8 = examine_and_correct_outliers(df_7, flagged_ids_lat, col='Latitude',  log_path=outpath)

In [ ]:
from utils.cleaning import final_coord_cleanup

df_9 = final_coord_cleanup(df_8, lat_col='Latitude', lon_col='Longitude')

# convert string "nan" (any case) to real NaN
df_10 = df_9.replace(r'^\s*nan\s*$', np.nan, regex=True)

# output remaining invalid tokens for inspection
invalid_lon, invalid_lat = save_invalid_coords(df_10, dir_path=Files)

In [ ]:
import csv
outpath = os.path.join(CSV, "logbooks_coords_cleaned.csv")
df_10.to_csv(outpath, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

### Unrealistic Coordinates
The lat/lon entries have been cleaned... but some inaccuracies often make it through this initial cleaning (due to inaccurate logkeepers or entry error). Here we flag unrealistic corrdinate jumps, points over land, and plot the new entries to visually check the ship trajectories:

Identification and flagging in column "coord_diff"

In [13]:
import pandas as pd, numpy as np, os

# Get current directory
current_directory = os.getcwd()
#print(current_directory)

# Specify the path to the folder you want to save data to 
Figures = os.path.join(current_directory, 'Figures')
Files = os.path.join(current_directory, 'Text Files')
CSV = os.path.join(current_directory, 'CSV Files')

# Create directories only if they don't exist
os.makedirs(Figures, exist_ok=True)
os.makedirs(Files, exist_ok=True)
os.makedirs(CSV, exist_ok=True)

In [12]:
from utils.cleaning import dms_to_decimal, add_decimal_columns, flag_unrealistic_coord_jumps, inspect_and_correct_logbook_flags

df_path = os.path.join(CSV, "logbooks_coords_cleaned.csv")
df = pd.read_csv(df_path, low_memory=False)

In [13]:
# hardcoding errors caught via flagging/inspection
NEW_COORD_FIXES = {
    # Charles Drew (Ship) 1846-1849
    32151: {"Latitude": "11 34 N"},   # '11 34 S' → '11 34 N'
    32492: {"Latitude": "50 48 N"},   # '50 48 S' → '50 48 N'
    35463: {"Latitude": "34 51 S"},   # '34 51 N' → '34 51 S'

    # Vineyard (Ship) 1844-1847
    27792: {"Latitude": "6 57 S"},    # '6 57 N' → '6 57 S'
    27840: {"Latitude": "35 09 N"},   # '35 09 S' → '35 09 N'

    # Hibernia (Ship) 1844-1846
    24277: {"Latitude": "49 32 N"},   # '49 32 S' → '49 32 N'
    24538: {"Longitude": "153 55 W"}, # '53 55 W' → '153 55 W'

    # Rodman (Bark) 1855-1859
    123733: {"Latitude": "35 38 S"},  # '25 38 S' → '35 38 S'
    123747: {"Longitude": "128 48 E"},# '148 48 E' → '128 48 E'
    125912: {"Latitude": "26 17 S"},  # '36 17 S' → '26 17 S'
    126775: {"Latitude": "34 03 S"},  # '24 03 S' → '34 03 S'

    # Amazon (1856-1860)
    124961: {"Longitude": "121 42 W"},# '131 42 W' → '121 42 W'
    125368: {"Longitude": "172 22 E"},# '142 22 E' → '172 22 E'
    126068: {"Latitude":  "0 34 N"},  # '0 34 S' → '0 34 N'
    126069: {"Latitude":  "1 10 N"},  # '1 10 S' → '1 10 N'
    126070: {"Latitude":  "1 39 N"},  # '1 39 S' → '1 39 N'
    126071: {"Latitude":  "2 55 N"},  # '2 55 S' → '2 55 N'
    126072: {"Latitude":  "4 27 N"},  # '4 27 S' → '4 27 N'
    126073: {"Latitude":  "6 33 N"},  # '6 33 S' → '6 33 N'
    126246: {"Latitude":  "8 37 N"},  # '8 37 S' → '8 37 N'
    124847: {"Latitude": "47 18 S"},  # '47 18 N' → '47 18 S'

    # Young Phenix (1867-1871)
    122266: {"Latitude": np.nan},     # '32 18 S' → nan

    # Francis Allyn (Schooner) 1891-…
    126528: {"Longitude": np.nan},    # '49 15 E' → nan
    127504: {"Longitude": "57 02 E"}, # '47 02 E' → '57 02 E'
    128855: {"Longitude": "49 33 E"}, # '59 33 E' → '49 33 E'
    128985: {"Longitude": "47 48 W"}, # '57 48 W' → '47 48 W'

    # Mary Frazier (Bark) 1865-1867
    104442: {"Longitude": "60 30 W"}, # '40 30 W' → '60 30 W'
    110714: {"Latitude": "19 22 S"},  # '29 22 S' → '19 22 S'

    # Jason (Ship) 1846-1848
    119221: {"Longitude": "50 29 W"}, # '30 29 W' → '50 29 W'

    # Louvre (Ship) 1842
    98864:  {"Longitude": "72 18 E"}, # '72 18 W' → '72 18 E'

    # Eagle (Ship) 1844-1848
    21394:  {"Latitude": "44 40 S"},  # '54 40 S' → '44 40 S'

    # Navy (Ship) 1851-1855
    102703: {"Longitude": "121 27 W"},# '111 27 W' → '121 27 W'

    # Awashonks (Ship) 1836-1840
    103564: {"Latitude": "43 00 S"},  # '53 00 S' → '43 00 S'

    # Gideon Howland (Ship) 1838-184…
    109071: {"Latitude": "39 27 S"},  # '59 27 S' → '39 27 S'

    #Elisha Dunbar (Bark) 1854-1858
    127997: {'Longitude': '64 14 E'}, # '64 14 W' → '64 14 E'
}

# apply corrections
from utils.cleaning import apply_coord_corrections
df = apply_coord_corrections(df, NEW_COORD_FIXES)

In [14]:
# add decimal columns to df
df = add_decimal_columns(df, lat_col='Latitude', lon_col='Longitude',
                         out_lat='Latitude_decimal', out_lon='Longitude_decimal')

# flag unrealistic jumps
df = flag_unrealistic_coord_jumps(
    df,
    time_col='Entry Date Time',
    logbook_col='LogBook ID',
    lat_col='Latitude_decimal',
    lon_col='Longitude_decimal',
    time_format='%Y-%m-%d %H:%M:%S',      # your current format
    time_delta_seconds=60*60*24*2,        # 2 days
    latlon_delta_deg=10.0,
    lon_delta_upper_limit=320.0
)

# See the flagged rows
suspects = df[df['coord_diff']]

In [15]:
# Which logbooks actually have flags?
flagged_books = df.loc[
    df['coord_diff']
    & df['Latitude_decimal'].notna()
    & df['Longitude_decimal'].notna(),
    'LogBook ID'
].value_counts()
#flagged_books

In [ ]:
logbook_id = flagged_books.index[6]
logbook_id

In [ ]:
lpath = os.path.join(Files, "coord_corrections_log.txt")

#inspecting the flagged coords
df = inspect_and_correct_logbook_flags(
    df,
    logbook_id=logbook_id,
    flagged_col="coord_diff",
    annotate=False,
    log_path = lpath,
    # upsert decimal columns after each edit
    recompute_decimal_after_each=True,
    recompute_decimal_fn=lambda d: d.assign(
        Latitude_decimal=d['Latitude'].apply(dms_to_decimal),
        Longitude_decimal=d['Longitude'].apply(dms_to_decimal)
    ),
    # re-run flagger after each edit
    reflag_after_each=True,
    reflag_fn=lambda d: flag_unrealistic_coord_jumps(
        d,
        time_col='Entry Date Time',
        logbook_col='LogBook ID',
        lat_col='Latitude_decimal',
        lon_col='Longitude_decimal',
        time_format='%Y-%m-%d %H:%M:%S',
        time_delta_seconds=60*60*24*2,
        latlon_delta_deg=10.0,
        lon_delta_upper_limit=320.0
    ),
)

In [18]:
import csv
outpath = os.path.join(CSV, "logbooks_flagged_cleaned.csv")
df.to_csv(outpath, index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

need bulk edit and set both to nan features
redo amazon, young phoenix, falcon, good return when implemented

redo 109071, 109072 Gideon Howland

remove alaska, president, ___, Congress (Ship) 1857-1859 for now

# Plotting the data

Lets plot all new entries that have not been visually cleaned in previous exports:

In [8]:
#Inspect trajectories of all new points to make sure flagging didnt miss anything

import pandas as pd, os, numpy as np

# Load both exports
df_new = pd.read_csv('/home/finn.wimberly/Documents/whaling_logs/20250728_export/CSV Files/logentries-export-2025-07-28.csv',  low_memory=False)
df_old = pd.read_csv('/home/finn.wimberly/Documents/whaling_logs/20250606_export/logentries-export-2025-06-11.csv', low_memory=False)

df_old['Entry Date'] = pd.to_datetime(df_old['Entry Date'], errors='coerce', format='mixed')
df_new['Entry Date'] = pd.to_datetime(df_new['Entry Date'], errors='coerce', format='mixed')

df_old['Entry Date'] = df_old['Entry Date'].dt.normalize()
df_new['Entry Date'] = df_new['Entry Date'].dt.normalize()

df_old_filled = df_old.fillna('##MISSING##')
df_new_filled = df_new.fillna('##MISSING##')

# Mapping of truncated names in new df -> full names
logbook_overwrite_map = {
    "A. M. Nicholson (schooner) 190…": "A. M. Nicholson (schooner) 1909-1910",
    "Francis Allyn (Schooner) 1891-…": "Francis Allyn (Schooner) 1891-1893",
    "Governor Hopkins (Brig) 1839-1…": "Governor Hopkins (Brig) 1839-1840",
    "George Washington (Bark) 1837-…": "George Washington (Bark) 1837-1840",
    "Gideon Howland (Ship) 1838-184…": "Gideon Howland (Ship) 1838-1842",
    "Gideon Howland (ship) 1836-183…": "Gideon Howland (ship) 1836-1838",
    "Walter Irving (Schooner) 1852-…": "Walter Irving (Schooner) 1852-1853",
    "Walter Irving (Schooner) 1856-…": "Walter Irving (Schooner) 1856-1857",
    "Walter Irving (Schooner) 1855-…": "Walter Irving (Schooner) 1855-1856",
    "General Jackson (ship) 1836-18…": "General Jackson (ship) 1836-1839",
    "Walter Irving (Schooner) 1854-…": "Walter Irving (Schooner) 1854-1855",
    "Walter Irving (Schooner) 1853-…": "Walter Irving (Schooner) 1853-1854",
    #"Gage H. Phillips (Schooner) 18…": "<FILL IN FULL NAME>",  # placeholder
    "Sally Anne (ship) & Seine…": "Sally Anne (ship) & Seine (bark) 1835-1838",
    "C.C. Comstock (Schooner) 1865-…": "C.C. Comstock (Schooner) 1865-1866",
    "George Clinton (ship) 1834-183…": "George Clinton (ship) 1834-1837",
    "Charles Phelps (Ship) 1850-185…": "Charles Phelps (Ship) 1850-1853",
    "Abraham Barker (Ship) 1850-185…": "Abraham Barker (Ship) 1850-1853",
    "Henry Kneeland (ship) 1848-185…": "Henry Kneeland (ship) 1848-1851",
    "Good Return II (ship) 1833-183…": "Good Return II (ship) 1833-1834",
    "Charles and Henry (ship) 1833-…": "Charles and Henry (ship) 1833-1836",
    "Bartholomew Gosnold (Bark) 188…": "Bartholomew Gosnold (Bark) 1881-1885",
    "Leonidas (ship) Journal 1830-1…": "Leonidas (ship) Journal 1830-1833",
    "Eunice H. Adams (Brig) 1887-18…": "Eunice H. Adams (Brig) 1887-1890",
    "Governor Carver (ship) 1857-18…": "Governor Carver (ship) 1857-1859",
    "Charles Phelps (Ship) 1844-184…": "Charles Phelps (Ship) 1844-1847",
    "Clifford Wayne (ship) 1844-184…": "Clifford Wayne (ship) 1844-1847",
    "Charles Phelps (ship) 1842-184…": "Charles Phelps (ship) 1842-1844",
    "Thomas Winslow (Brig) 1846-184…": "Thomas Winslow (Brig) 1846-1847",
    "Charles W. Morgan (bark) 1911-…": "Charles W. Morgan (bark) 1911-1912",
}

# Apply the mapping to overwrite in df_new_filled
df_new_filled["LogBook ID"] = df_new_filled["LogBook ID"].replace(logbook_overwrite_map)

In [ ]:
# Find new rows
new_rows = df_new_filled.merge(df_old_filled.drop_duplicates(), how='left', indicator=True)
new_rows = new_rows[new_rows['_merge'] == 'left_only'].drop(columns=['_merge'])

# Display result
print(f"{len(new_rows)} new row(s) found.")
#display(new_rows)

In [ ]:
# Extract unique values from the 'LogBook ID' column
logbooks_new_entries = new_rows['LogBook ID'].unique()
logbooks_new_entries = logbooks_new_entries[logbooks_new_entries != "Gage H. Phillips (Schooner) 18…"]

# Convert to a sorted list (optional)
logbooks_new_entries = sorted(logbooks_new_entries)

# Display result
print(f"{len(logbooks_new_entries)} logbooks with new entries found:")
for logbook in logbooks_new_entries:
    print(logbook)

In [ ]:
# Get sets of unique logbook IDs
new_logbook_ids = set(df_new_filled['LogBook ID'].unique())
old_logbook_ids = set(df_old_filled['LogBook ID'].unique())

# Subtract to find newly added logbooks
newly_added_logbooks = sorted(new_logbook_ids - old_logbook_ids)

# Display the result
print(f"{len(newly_added_logbooks)} new logbooks found:")
for logbook in newly_added_logbooks:
    print(logbook)

In [14]:
df_path = os.path.join(CSV, "logbooks_flagged_cleaned.csv")
df = pd.read_csv(df_path, low_memory=False)

In [ ]:
from utils.cleaning import plot_new_entries

fig, axes, info = plot_new_entries(
    df_corrected=df,
    new_rows=new_rows,
    figures_dir=None, #set to Figures to save
    export_label="2025-08-12 export",  
    plot_scope="all_from_new_logbooks",   
    ncols=2,
    exclude_logbooks=[
        "Gage H. Phillips (Schooner) 18…", "TEST TEST HG JULY 2025"  # exclusions logbook that isn't new
    ],
)
print(info)

In [ ]:
from utils.cleaning import plot_logbook

_ = plot_logbook(
    df, "Elisha Dunbar (Bark) 1854-1858",
    years=[1855],                
    annotate=True,                
    annotate_field="ID",
    annotate_max=500,  
    figures_dir = Figures,
    filename="inspect_elisha.png",
    dpi=300,
)

In [ ]:
#df[df['ID'] == 122798]
row_idx = df.index[df['ID'] == 127979][0]

from utils.cleaning import correct_coord
from utils.cleaning import dms_to_decimal

lpath = os.path.join(Files, "coord_corrections_log.txt")
correct_coord(df, row_idx, col = 'Latitude', log_path=lpath, force_both=False)

# recompute decimal degrees for adjusted row
df.loc[row_idx, 'Latitude_decimal']  = dms_to_decimal(df.loc[row_idx, 'Latitude'])
df.loc[row_idx, 'Longitude_decimal'] = dms_to_decimal(df.loc[row_idx, 'Longitude'])

#### After inspecting all new entries and making corrections, we are done. Save the cleaned dataset:

In [21]:
# Work with a copy to avoid SettingWithCopyWarning
temp_df = df.copy()

# Convert to datetime safely
temp_df['Entry Date Time'] = pd.to_datetime(temp_df['Entry Date Time'], errors='coerce')

# List of logbooks to exclude completely
exclude_logbooks = [
    'Alaska (Bark) 1880-1884',
    'Mercator(Ship) 1840-1843',
    "TEST TEST HG JULY 2025"
]

# Define rows to exclude
exclude_rows = (
    ((temp_df['LogBook ID'] == 'Howard (Ship) 1838-1841') & (temp_df['ID'] == 109907)) |
    (temp_df['LogBook ID'].isin(exclude_logbooks))  # <- NEW condition
)

# Apply filtering and save
current_directory = os.getcwd()
PKL = os.path.join(current_directory, "PKL")
os.makedirs(PKL, exist_ok=True)

df_cleaned = temp_df[~exclude_rows]
df_cleaned.to_csv(f'{CSV}/final_logbooks.csv', index=False)
df_cleaned.to_pickle(f"{PKL}/Tier1.pkl")